In [58]:
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА СЫРЫХ ДАННЫХ
# Загружаем выгрузку из системы Росстата (абсолютные данные по миграции)
df = pd.read_excel('data/raw/23320000100030200007_Миграционный_прирост_(убыль)_населения,_абсолютные_данные.xlsx')
df_clean = df.copy()

# 2. ОБРАБОТКА ЗАГОЛОВКОВ (ПАРСИНГ ВРЕМЕННЫХ РЯДОВ)
# Извлекаем названия лет из первой строки и очищаем их от лишних символов (" г.")
raw_years = df_clean.iloc[0, 2:].dropna().values
clean_years = [str(y).replace(' г.', '').strip() for y in raw_years]

# Формируем корректные названия столбцов
new_columns = ['Регион', 'Код'] + clean_years
df_clean.columns = new_columns

# 3. ТИПИЗАЦИЯ ДАННЫХ
# Принудительно приводим данные по годам к числовому формату (ошибки парсинга станут NaN)
for col in clean_years:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 4. НОРМАЛИЗАЦИЯ ТЕРРИТОРИАЛЬНЫХ КОДОВ (OKATO/OKTMO)
# Функция для приведения кодов регионов к единому формату (первые 3 цифры)
def normalize_code(val):
    s = str(val).split('.')[0].strip()
    return s[:3] if len(s) >= 5 else s

# Применяем нормализацию кодов для возможности последующего объединения таблиц
df_clean['Код'] = df_clean['Код'].apply(normalize_code)
df_clean['Код'] = pd.to_numeric(df_clean['Код'], errors='coerce

# Финальная очистка: удаляем записи, где полностью отсутствуют значения в числовых колонках
df_clean = df_clean.dropna(subset=clean_years, how='all')

# 5. ЭКСПОРТ ПРОМЕЖУТОЧНОГО РЕЗУЛЬТАТА
# Сохраняем очищенные данные в CSV для дальнейшей трансформации
df_clean.head()
#df_clean.to_csv('data/input_clean/test.csv', index=False, sep=';', encoding='utf-8-sig')

In [60]:
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА И ПЕРВИЧНАЯ КОПИЯ
# Загружаем детальные данные (пол + возраст + потоки)
df = pd.read_excel('data/raw/23320000100030200001_Миграционный_прирост_населения_по_полу,_возрасту_и_потокам_передвижения.xlsx')
df_clean = df.copy()

# 2. ОЧИСТКА ЗАГОЛОВКОВ ВРЕМЕННЫХ РЯДОВ
# Извлекаем года, убираем лишние приставки " г." и формируем названия колонок
raw_years = df_clean.iloc[0, 2:].dropna().values
clean_years = [str(y).replace(' г.', '').strip() for y in raw_years]
new_columns = ['Регион', 'Код'] + clean_years
df_clean.columns = new_columns

# 3. ИЗВЛЕЧЕНИЕ СКРЫТЫХ КАТЕГОРИЙ (МЕТОД FFILL)
# Данные в Excel идут «слоями». Мы создаем новые признаки и заполняем их сверху вниз
df_clean['Пол'] = None
df_clean['Местность'] = None

# Ищем маркеры пола и типа местности в текстовом описании строк
df_clean.loc[df_clean['Регион'].str.contains('Женщины|Мужчины', na=False), 'Пол'] = df_clean['Regon']
df_clean.loc[df_clean['Регион'].str.contains('местность', na=False), 'Местность'] = df_clean['Регион']

# Распространяем найденные категории на все нижеследующие строки до появления новой категории
df_clean['Пол'] = df_clean['Пол'].ffill()
df_clean['Местность'] = df_clean['Местность'].ffill()

# 4. ТИПИЗАЦИЯ И ЧИСТКА КОДОВ
# Переводим данные по годам в числовой формат
for col in clean_years:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Нормализуем коды регионов (приведение к 3-значному стандарту для джойна)
def normalize_code(val):
    if pd.isna(val): return val
    s = str(val).split('.')[0].strip()
    return s[:3] if len(s) >= 5 else s
df_clean['Код'] = df_clean['Код'].apply(normalize_code)

# Удаляем «пустые» строки, не содержащие никакой численной информации за период
mask = df_clean[clean_years].any(axis=1)
df_clean = df_clean[mask].reset_index(drop=True)

# 5. ФИЛЬТРАЦИЯ И АГРЕГАЦИЯ
# Оставляем только строки с валидными кодами регионов (отсекаем текстовые подзаголовки)
df_data = df_clean[df_clean['Код'].astype(str).str.contains(r'\d', na=False)].copy()

# Финальное группирование для устранения дублей после обработки иерархии
df_final = df_data.groupby(['Регион', 'Код', 'Пол', 'Местность'], sort=False)[clean_years].sum(min_count=1).reset_index()

# 6. СОХРАНЕНИЕ
#df_final.to_csv('data/input_clean/test.csv', index=False, sep=';', encoding='utf-8-sig')

In [59]:
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА И ПОДГОТОВКА
df = pd.read_csv('data/input_clean/test.csv', sep=';', encoding='utf-8-sig')
# Определяем список колонок-лет (все, кроме Региона и Кода)
clean_years = df.columns[2:].tolist()

# Убеждаемся, что все данные в числовом формате (ошибки станут NaN)
df[clean_years] = df[clean_years].apply(pd.to_numeric, errors='coerce')

# 2. СТРАТЕГИЯ ВОССТАНОВЛЕНИЯ ПРОПУСКОВ (TIME-SERIES IMPUTATION)

# Шаг А: Линейная интерполяция
# Заполняем пропуски внутри временного ряда (например, если есть 2015 и 2017, высчитываем среднее для 2016)
df[clean_years] = df[clean_years].interpolate(axis=1, method="linear")

# Шаг Б: Метод ближайших соседей (Forward/Backward Fill)
# Если пропуск на краях (первый или последний год), копируем значение из соседнего года
# limit=1 ограничивает заполнение только на один шаг, чтобы не раздувать данные искусственно
df[clean_years] = df[clean_years].ffill(axis=1, limit=1).bfill(axis=1, limit=1)

# Шаг В: Заполнение по среднему значению региона
# Если данные отсутствуют за несколько лет подряд, используем среднее арифметическое этого региона за весь период
df[clean_years] = df[clean_years].apply(lambda row: row.fillna(row.mean()), axis=1).round(0)

# 3. ЭКСПОРТ ОЧИЩЕННОГО ТРЕНДА
# Сохраняем финальный набор данных для миграционного анализа
#df.to_csv('data/input_clean/clean_migration_increase.csv', index=False, sep=';', encoding='utf-8-sig')

In [88]:
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА ДАННЫХ ПО КРУПНЫМ ГОРОДАМ
# Загружаем статистику прибывших в города с населением свыше 100 тыс. человек
df = pd.read_csv('data/input_clean/clean_count_of_arrivals_in_city_with_population_highest_100_thousand.csv', 
                 sep=';', encoding='utf-8-sig')

# Определяем временной интервал (колонки с годами)
clean_years = df.columns[2:].tolist()

# 2. ВОССТАНОВЛЕНИЕ КОНТЕКСТА (МЕТОД FFILL)
# В исходном Excel типы местности/категории часто указаны одной строкой над данными.
# Создаем отдельный признак и протягиваем его вниз до следующего заголовка.
df['Местность'] = None
mask = df['Регион'].str.contains('местность|население', na=False, case=False)
df.loc[mask, 'Местность'] = df['Regon']
df['Местность'] = df['Местность'].ffill()

# 3. ФИЛЬТРАЦИЯ ПУСТЫХ ЗАПИСЕЙ
# Удаляем строки-заголовки и пустые записи, где нет численных данных за выбранные годы
df = df.dropna(subset=clean_years, how='all')

# 4. РЕОРГАНИЗАЦИЯ СТРУКТУРЫ
# Переупорядочиваем столбцы: выносим технические признаки (Код, Местность) в начало
cols = ['Регион', 'Код', 'Местность'] + clean_years
df = df[cols]

# 5. СОХРАНЕНИЕ ОЧИЩЕННОГО СРЕЗА
# Эти данные станут основой для анализа привлекательности региональных центров
# df.to_csv('data/input_clean/clean_count_of_arrivals_in_city_with_population_highest_100_thousand.csv', 
#          index=False, sep=';', encoding='utf-8-sig')

In [63]:
from scipy import stats
import pandas as pd
import numpy as np

# 1. ЗАГРУЗКА И ПОДГОТОВКА
# Загружаем данные по миграционным потокам в разрезе пола
df = pd.read_csv('data/input_clean/clean_migration_increase_per_gender.csv', sep=';', encoding='utf-8-sig')
clean_years = df.columns[4:].tolist()

# 2. ЛОГАРИФМИЧЕСКОЕ ПРЕОБРАЗОВАНИЕ
# В данных есть «гиганты» (Москва) и малые регионы. Логарифмирование сглаживает разброс,
# позволяя корректно сравнивать их в рамках одной модели.
# np.log1p используется после нормализации значений к положительному диапазону.
df_log = np.log1p(df[clean_years] - df[clean_years].min().min() + 1)

# 3. ПОИСК АНОМАЛИЙ ЧЕРЕЗ Z-SCORE
# Рассчитываем Z-оценку: количество стандартных отклонений от среднего для каждого значения.
# Формула: $z = \frac{x - \mu}{\sigma}$
z_scores_log = np.abs(stats.zscore(df_log))

# Устанавливаем порог в 3 сигма (правило трех сигм). 
# Все, что выше 3, — статистические аномалии (регионы-лидеры с экстремальными потоками).
outliers_mask = (z_scores_log > 3).any(axis=1)

# Вывод регионов-аномалий для визуального контроля
print("Обнаруженные регионы-лидеры (аномалии):")
print(df[outliers_mask])

# 4. РАЗДЕЛЕНИЕ ДАТАСЕТА
# Выделяем «лидеров» в отдельный файл для специального анализа (гипотеза №1)
df_outliers = df[outliers_mask].copy()
# Оставляем «типичные» регионы для общего статистического анализа
df_normal = df[~outliers_mask].copy()

# 5. ЭКСПОРТ
# df_outliers.to_csv('data/migrations_absolute_counts_outliers_leaders.csv', index=False, sep=';', encoding='utf-8-sig')
# df_normal.to_csv('data/clean_count_of_arrivals_per_gender.csv', index=False, sep=';', encoding='utf-8-sig')

Обнаруженные регионы-лидеры (аномалии):
                                                Регион  Код      Пол  \
0                                 Российская Федерация  643  Женщины   
1                        Центральный федеральный округ   30  Женщины   
11                                  Московская область  460  Женщины   
19   Город Москва столица Российской Федерации горо...  450  Женщины   
20                   Северо-Западный федеральный округ   31  Женщины   
35              Южный федеральный округ (с 29.07.2016)   40  Женщины   
110                               Российская Федерация  643  Женщины   
159                      Приволжский федеральный округ   33  Женщины   
182                        Сибирский федеральный округ   35  Женщины   
207                               Российская Федерация  643  Мужчины   
208                      Центральный федеральный округ   30  Мужчины   
226  Город Москва столица Российской Федерации горо...  450  Мужчины   
288                     

In [44]:
import pandas as pd

# 1. ЗАГРУЗКА ДАННЫХ
# Загружаем подготовленный датасет с миграционными потоками по полу
df = pd.read_csv('data/input_clean/clean_count_of_migrations_per_gender.csv', sep=';', encoding='utf-8-sig')

# 2. ФИЛЬТРАЦИЯ АГРЕГИРОВАННЫХ ПОКАЗАТЕЛЕЙ
# Создаем маску для поиска строки с суммарными данными по всей стране
# Использование .str.strip() важно, так как в названиях могут быть лишние пробелы
russia_mask = df['Регион'].str.strip() == 'Российская Федерация'

# Разделяем датасет: 
# Можно сохранить итоги по РФ отдельно для макро-анализа (опционально)
# df_russia_totals = df[russia_mask].copy()

# Оставляем только субъекты РФ для детального регионального анализа
df_regions_only = df[~russia_mask].copy()

# 3. ЭКСПОРТ ОЧИЩЕННОГО ДАТАСЕТА
# Перезаписываем файл, теперь в нем содержатся только сопоставимые данные по регионам
# df_regions_only.to_csv('data/input_clean/clean_count_of_migrations_per_gender.csv', index=False, sep=';', encoding='utf-8-sig')